# Student Performance Datasets — Train / Validation / Test Preparation

This notebook adapts the workflow structure of the supplied Qwen/TinyStories notebook to the two **tabular student-performance datasets**.

**Important:** The supplied Qwen notebook is a text-language-model project. Its tokenizer, token IDs, `.bin` files, RoPE, transformer architecture and text-generation loop are not suitable for these CSV datasets. This notebook uses pandas and scikit-learn to prepare tabular data instead.

## Deliverables
- Two datasets kept separate (their columns and meanings differ).
- Reproducible 70% train / 15% validation / 15% test splits.
- Six Apache Parquet files.
- Upload to `monika-p/students-performance-splits` on Hugging Face.
- Final repository verification.

The notebook expects the two CSV files to be in the same folder as this notebook:
- `StudentPerformance.csv` — Exams dataset (expected 1,000 rows × 8 columns)
- `StudentsPerformance.csv` — 1-million-row dataset (expected 1,000,000 rows × 6 columns)

Do not rename these files unless you also update the filenames in the configuration cell.

## 1. Install required packages

Run this cell once. If Jupyter asks for a kernel restart after installation, restart it and run the notebook cells again from the top.

In [ ]:
%pip install -q --upgrade pandas numpy scikit-learn pyarrow huggingface_hub

## 2. Import libraries and set reproducibility

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXAMS_CSV = "StudentPerformance.csv"
MILLION_CSV = "StudentsPerformance.csv"
OUTPUT_DIR = Path("parquet_output")

REPO_ID = "monika-p/students-performance-splits"
REPO_TYPE = "dataset"

print("Libraries imported.")
print("Working directory:", Path.cwd())

## 3. Check that the two source CSV files exist

The filenames are intentionally mapped as follows, based on the file inspection in the earlier workflow:
- `StudentPerformance.csv` = 1,000-row Exams dataset
- `StudentsPerformance.csv` = 1,000,000-row dataset

If your files have different names, change the two filename variables in Section 2.

In [ ]:
for filename in [EXAMS_CSV, MILLION_CSV]:
    path = Path(filename)
    print(f"{filename}: {'FOUND' if path.is_file() else 'NOT FOUND'}")
    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot find {filename}. Upload it to the same folder as this notebook "
            "or correct the filename in Section 2."
        )

## 4. Load both datasets and inspect their structure

In [ ]:
df_exams = pd.read_csv(EXAMS_CSV)
df_million = pd.read_csv(MILLION_CSV)

print("EXAMS DATASET SHAPE:", df_exams.shape)
print("EXAMS COLUMNS:", df_exams.columns.tolist())
display(df_exams.head())

print("\nMILLION DATASET SHAPE:", df_million.shape)
print("MILLION COLUMNS:", df_million.columns.tolist())
display(df_million.head())

## 5. Validate expected schemas and inspect data quality

This cell checks the columns and reports missing values and duplicate rows. It does not silently delete records or fill missing values. If any issue appears, inspect it before proceeding.

In [ ]:
expected_exams_columns = {
    "gender", "race/ethnicity", "parental level of education", "lunch",
    "test preparation course", "math score", "reading score", "writing score"
}
expected_million_columns = {
    "student_id", "weekly_self_study_hours", "attendance_percentage",
    "class_participation", "total_score", "grade"
}

assert set(df_exams.columns) == expected_exams_columns, (
    "The Exams dataset columns do not match the expected 8-column schema. "
    f"Found: {df_exams.columns.tolist()}"
)
assert set(df_million.columns) == expected_million_columns, (
    "The 1-million dataset columns do not match the expected 6-column schema. "
    f"Found: {df_million.columns.tolist()}"
)

print("EXAMS missing values:\n", df_exams.isna().sum())
print("EXAMS duplicate rows:", df_exams.duplicated().sum())

print("\nMILLION missing values:\n", df_million.isna().sum())
print("MILLION duplicate rows:", df_million.duplicated().sum())
print("MILLION unique student IDs:", df_million["student_id"].nunique())

if df_exams.isna().any().any() or df_million.isna().any().any():
    raise ValueError("Missing values detected. Review and decide how to handle them before splitting.")

print("\nSchema and missing-value checks passed.")

## 6. Create reproducible train / validation / test splits

Each dataset is split separately because the two datasets contain different variables and represent different data structures.

- Train: 70%
- Validation: 15%
- Test: 15%
- Random seed: 42

For the 1-million-row dataset, `grade` is used only as a stratification column so that grade proportions remain similar across the splits. The grade column remains in the saved files.

In [ ]:
# Exams dataset: 70% train, 15% validation, 15% test
exams_train, exams_temp = train_test_split(
    df_exams,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)
exams_validation, exams_test = train_test_split(
    exams_temp,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# Million-row dataset: stratified by grade
million_train, million_temp = train_test_split(
    df_million,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=df_million["grade"]
)
million_validation, million_test = train_test_split(
    million_temp,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=million_temp["grade"]
)

print("Both datasets were split.")

## 7. Verify split counts and row totals

In [ ]:
split_summary = pd.DataFrame([
    ["Exams", len(df_exams), len(exams_train), len(exams_validation), len(exams_test)],
    ["Million", len(df_million), len(million_train), len(million_validation), len(million_test)]
], columns=["Dataset", "Original rows", "Train rows", "Validation rows", "Test rows"])

display(split_summary)

assert len(exams_train) + len(exams_validation) + len(exams_test) == len(df_exams)
assert len(million_train) + len(million_validation) + len(million_test) == len(df_million)

# Expected counts for the two files inspected in the earlier workflow
assert (len(exams_train), len(exams_validation), len(exams_test)) == (700, 150, 150)
assert (len(million_train), len(million_validation), len(million_test)) == (700000, 150000, 150000)

print("All split counts are correct.")

## 8. Check grade proportions in the 1-million-row splits

The proportions should be very close because the split was stratified by `grade`.

In [ ]:
grade_distribution = pd.concat({
    "Original": df_million["grade"].value_counts(normalize=True).sort_index(),
    "Train": million_train["grade"].value_counts(normalize=True).sort_index(),
    "Validation": million_validation["grade"].value_counts(normalize=True).sort_index(),
    "Test": million_test["grade"].value_counts(normalize=True).sort_index()
}, axis=1).fillna(0)

display((grade_distribution * 100).round(2))

## 9. Save all six splits as Apache Parquet

Parquet is a columnar storage format suitable for analytics and machine-learning data pipelines. The original CSV files are not modified.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parquet_datasets = {
    "exams_train.parquet": exams_train,
    "exams_validation.parquet": exams_validation,
    "exams_test.parquet": exams_test,
    "million_train.parquet": million_train,
    "million_validation.parquet": million_validation,
    "million_test.parquet": million_test,
}

for filename, dataframe in parquet_datasets.items():
    output_path = OUTPUT_DIR / filename
    dataframe.to_parquet(output_path, index=False)
    print(f"Saved: {output_path} | rows={len(dataframe):,} | columns={len(dataframe.columns)}")

print("\nAll six Parquet files saved.")

## 10. Read the Parquet files back and verify them

In [ ]:
expected_rows = {
    "exams_train.parquet": 700,
    "exams_validation.parquet": 150,
    "exams_test.parquet": 150,
    "million_train.parquet": 700000,
    "million_validation.parquet": 150000,
    "million_test.parquet": 150000,
}

verification_rows = []

for filename, expected_count in expected_rows.items():
    path = OUTPUT_DIR / filename
    check_df = pd.read_parquet(path)
    actual_count = len(check_df)
    assert actual_count == expected_count, (
        f"{filename}: expected {expected_count} rows, found {actual_count}"
    )
    verification_rows.append({
        "file": filename,
        "rows": actual_count,
        "columns": len(check_df.columns),
        "size_MB": round(path.stat().st_size / (1024 * 1024), 2),
        "readable": True
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)
print("All six Parquet files were read back and verified.")

## 11. Authenticate with Hugging Face

The repository is `monika-p/students-performance-splits`.

1. Create a Hugging Face access token with **Write** permission.
2. Run the cell below.
3. Paste the token into the secure prompt when requested. Do not put the token directly in notebook code, and do not share it in screenshots or GitHub.

In [ ]:
from huggingface_hub import login, HfApi

login()  # Paste your WRITE token into the secure prompt

api = HfApi()
account = api.whoami()
print("Authenticated Hugging Face account:", account["name"])

if account["name"] != "monika-p":
    raise PermissionError(
        f"Logged in as {account['name']}, but this notebook expects monika-p. "
        "Authenticate with the correct account before uploading."
    )

## 12. Create the dataset repository if needed

This creates the repository only if it does not already exist. It does not delete an existing repository.

In [ ]:
api.create_repo(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    private=False,
    exist_ok=True
)
print("Repository is ready:", f"https://huggingface.co/datasets/{REPO_ID}")

## 13. Upload the six Parquet files to Hugging Face

In [ ]:
# Confirm all six expected files exist before uploading
for filename in expected_rows:
    if not (OUTPUT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing file: {OUTPUT_DIR / filename}")

upload_commit = api.upload_folder(
    folder_path=str(OUTPUT_DIR),
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    allow_patterns=["*.parquet"],
    commit_message="Upload student performance train validation and test Parquet splits"
)

print("Upload commit:", upload_commit)
print("Upload request completed.")

## 14. Verify the files on Hugging Face

In [ ]:
remote_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)
remote_parquet_files = sorted(
    filename for filename in remote_files
    if filename.endswith(".parquet")
)

print("Parquet files found in the Hugging Face repository:")
for filename in remote_parquet_files:
    print("✓", filename)

assert set(remote_parquet_files) == set(expected_rows.keys()), (
    "The remote repository does not contain exactly the six expected Parquet files. "
    f"Found: {remote_parquet_files}"
)

print("\nSUCCESS: All six Parquet files are present in the Hugging Face repository.")
print(f"Repository URL: https://huggingface.co/datasets/{REPO_ID}/tree/main")

## 15. Final project summary

At the end of this notebook, the project should contain:

| Dataset | Train | Validation | Test |
|---|---:|---:|---:|
| Exams | 700 | 150 | 150 |
| 1-million-row dataset | 700,000 | 150,000 | 150,000 |

**Hugging Face repository:**  
https://huggingface.co/datasets/monika-p/students-performance-splits

**Files uploaded:**
- `exams_train.parquet`
- `exams_validation.parquet`
- `exams_test.parquet`
- `million_train.parquet`
- `million_validation.parquet`
- `million_test.parquet`

Save this notebook after successful execution. Before publishing it on GitHub, clear notebook outputs if they contain any private information, and never include an access token.